# Milestone 2 — MERFISH spatial receptor maps

Per-gene coronal, sagittal, and axial scatter plots in **CCF coordinates** (`x_ccf`, `y_ccf`, `z_ccf`).
Section coordinates (`x_section`, etc.) are retained in metadata but not used for plotting.

Genes outside the ~500-gene panel use the imputed matrix when `use_imputed_merfish: true` (~50 GB download on first access).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings

from src.config import DEFAULT_OUTPUT_DIR, load_config, start_run
from src.data_loaders import (
    get_abc_cache,
    load_merfish_cell_metadata,
    check_gene_availability,
    load_single_gene_merfish,
)
from src.plotting import plot_spatial, plot_family_spatial_panel

In [ ]:
EXPLORATION_ROOT = DEFAULT_OUTPUT_DIR

CONFIG_PATH = PROJECT_ROOT / "receptor_query_config.yaml"
config = load_config(CONFIG_PATH)
OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset=config["data"]["merfish_dataset"],
    exploration_root=EXPLORATION_ROOT,
    notebook="02_merfish_spatial",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)
print(f"Run dir: {OUTPUT_DIR}")
print(f"Manifest: {OUTPUT_DIR / 'run_manifest.json'}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

In [ ]:
cell_meta = load_merfish_cell_metadata(cache, config)
print(f"MERFISH cells with CCF coords: {len(cell_meta):,}")
print("Coordinate columns:", [c for c in cell_meta.columns if "ccf" in c or "section" in c])

In [ ]:
projections = ["coronal", "sagittal", "axial"]
family_results: dict[str, dict] = {fam: {} for fam in config["_families"]}

for gene in config["_all_genes"]:
    status = check_gene_availability(cache, gene, config)
    if status == "missing":
        warnings.warn(f"{gene}: not in MERFISH panel or imputed set; skipping.")
        continue

    expr, source = load_single_gene_merfish(cache, gene, config)
    if expr is None:
        warnings.warn(f"{gene}: could not load expression; skipping.")
        continue

    family = config["_genes_flat"][gene]
    family_results[family][gene] = {"expression": expr, "source": source}

    for proj in projections:
        out = OUTPUT_DIR / f"spatial_{gene}_{proj}.png"
        plot_spatial(
            cell_meta,
            expr,
            gene,
            proj,
            config,
            source_label=source,
            coord_prefix="ccf",
            save_path=out,
        )
        print(f"Saved {out} ({source})")

In [ ]:
for family in config["_families"]:
    results = family_results.get(family, {})
    if not results:
        continue
    path = plot_family_spatial_panel(results, family, cell_meta, config, output_dir=OUTPUT_DIR)
    if path:
        print(f"Saved family panel {path}")